# 01 · Threat modeling as code with OWASP `pytm`

**Objective (20 min):** express the RAG-agent architecture as Python, let `pytm` propose
threats, then show that *adding a control changes the model's output* — and end each selected
threat in an owner and a verification method.

The model lives in `rag_agent_tm.py` (≈90 lines). Read it before running: every element,
boundary, and flow is ordinary Python you can diff in a pull request.

In [ ]:
# --- Workshop bootstrap: run this cell first ------------------------------------
# JupyterLab starts every kernel inside the notebook's own folder. Move to the
# toolkit root so shared modules (demo_agent, workshop_utils) import and the
# _evidence/ output paths resolve, no matter where Jupyter was launched from.
import os, sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "workshop_utils.py").exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Workshop root:", ROOT)

In [ ]:
import json
import subprocess
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from workshop_utils import require_package, save_json

require_package("pytm")

# pytm parses sys.argv when the model is *executed*; when we import it we
# give it a clean argv so the notebook kernel's flags do not confuse it.
sys.argv = ["rag_agent_tm"]
sys.path.insert(0, str(ROOT / "01_Threat_Modeling"))
from rag_agent_tm import tm, llm, agent, flows  # noqa: E402

## 1. Resolve threats in-process

`tm.resolve()` evaluates every threat rule in pytm's library against every element and flow.
pytm 1.4 ships LLM/agent-specific rules (`LLM01`–`LLM09`) alongside the classic ones.

In [ ]:
tm.resolve()
findings = pd.DataFrame([
    {
        "threat_id": f.threat_id,
        "severity": f.severity,
        "target": str(f.target),
        "description": f.description,
    }
    for f in tm.findings
])
print(f"{len(findings)} findings across {findings['target'].nunique()} elements")
display(findings["severity"].value_counts().rename("count").to_frame())

llm_findings = findings[findings.threat_id.str.startswith("LLM")].sort_values("threat_id")
display(llm_findings)

## 2. Watch a control remove a threat

`LLM05 – Excessive Agency` fires because the LLM `hasAgentCapabilities`, `hasAccessToSensitiveSystems`,
and does **not** `implementsPOLP` (principle of least privilege). Module 03 builds exactly that control.
Flip the attributes and re-resolve: the threat model should change with the architecture.

In [ ]:
before = set(llm_findings.threat_id)

llm.controls.implementsPOLP = True       # typed proposals + scoped capabilities (module 03)
llm.hasContentFiltering = True           # input/output screening (modules 02, 04)
agent.validatesToolLaunchConfig = True   # tool config is validated before use

tm.findings = []          # discard the previous resolution
tm.resolve()
after = {f.threat_id for f in tm.findings if f.threat_id.startswith("LLM")}

print("LLM threats before controls:", sorted(before))
print("LLM threats after  controls:", sorted(after))
print("Removed:", sorted(before - after))
assert {"LLM05"} <= before - after, "implementsPOLP should have removed Excessive Agency"
assert "LLM03" in after, "Third-party data leakage still applies until we minimise/redact (module 05)"

## 3. Generate the data-flow diagram

`tm.dfd()` returns Graphviz DOT. Rendering needs the `dot` binary (optional); the text itself
is useful in CI and code review. Below we also derive a Mermaid diagram from the same model so it
renders inline in JupyterLab (≥ 4.1) and on GitHub.

In [ ]:
dot = tm.dfd()
out_dot = Path("_evidence/01_rag_agent_dfd.dot")
out_dot.parent.mkdir(exist_ok=True)
out_dot.write_text(dot, encoding="utf-8")
print(dot[:600], "...\n")
print(f"Wrote {out_dot} ({len(dot.splitlines())} lines). Render with: dot -Tpng {out_dot} -o dfd.png")

In [ ]:
def mermaid_from_model(flows) -> str:
    boundaries: dict[str, set[str]] = {}
    for f in flows:
        for el in (f.source, f.sink):
            b = getattr(el, "inBoundary", None)
            boundaries.setdefault(b.name if b else "(none)", set()).add(el.name)
    ident = lambda name: "n_" + "".join(ch if ch.isalnum() else "_" for ch in name)
    lines = ["flowchart LR"]
    for b, els in boundaries.items():
        lines.append(f'  subgraph "{b}"')
        lines += [f'    {ident(e)}["{e}"]' for e in sorted(els)]
        lines.append("  end")
    for i, f in enumerate(flows, 1):
        lines.append(f'  {ident(f.source.name)} -->|"{i}. {f.name}"| {ident(f.sink.name)}')
    return "\n".join(lines)

mermaid = mermaid_from_model(flows)
Path("_evidence/01_rag_agent_dfd.mmd").write_text(mermaid, encoding="utf-8")
display(Markdown("```mermaid\n" + mermaid + "\n```"))

## 4. The CLI is the same model

The file is executable. In CI you would run it with `--dfd`, `--seq`, or `--json` and check the
output into the repository so an architecture change shows up as a diff.

In [ ]:
result = subprocess.run(
    [sys.executable, "01_Threat_Modeling/rag_agent_tm.py", "--json", "_evidence/01_rag_agent_tm.json"],
    text=True, capture_output=True, check=False,
)
print(result.stdout[-1500:], result.stderr[-1500:])
assert result.returncode == 0, "pytm CLI failed"
print("JSON model written:", Path("_evidence/01_rag_agent_tm.json").stat().st_size, "bytes")

## 5. From findings to an owned backlog

Generated threats are *hypotheses*. Each one you keep must end in a prevention, a detection, an
executable verification, and an owner. Extend the rows below with at least one finding from step 1.

In [ ]:
backlog = [
    {
        "threat_ids": ["LLM02", "LLM08"],
        "unacceptable_outcome": "Secret or cross-tenant data disclosure",
        "attack_path": "Untrusted prompt -> retriever -> mixed-tenant chunk -> model response",
        "prevention": "Tenant filter enforced server-side; source trust metadata; no secrets in prompt context",
        "detection": "Canary and cross-tenant eval suite; retrieval trace sampling",
        "verification": "CI assertion: zero foreign-tenant document IDs in 1,000 adversarial retrievals",
        "owner": "RAG platform",
    },
    {
        "threat_ids": ["LLM05", "LLM09"],
        "unacceptable_outcome": "Unauthorized refund",
        "attack_path": "Prompt injection -> model emits tool call -> refund API",
        "prevention": "Typed proposal, amount policy, least-privilege token, human approval above INR 500",
        "detection": "Trace decision and approval ID; alert on policy bypass",
        "verification": "Integration test proves no committed refund without a valid approval record",
        "owner": "Agent platform",
    },
    {
        "threat_ids": ["LLM03"],
        "unacceptable_outcome": "Sensitive data copied to model vendor or logs",
        "attack_path": "User PII -> prompt/trace exporter -> external processor",
        "prevention": "Data minimization and pre-egress redaction",
        "detection": "PII scanner on sampled egress and telemetry",
        "verification": "Known email/phone corpus produces zero raw values in exported spans",
        "owner": "Privacy engineering",
    },
]
required = {"threat_ids", "unacceptable_outcome", "attack_path", "prevention", "detection", "verification", "owner"}
assert len(backlog) >= 3
assert all(required <= row.keys() and all(str(row[k]).strip() for k in required) for row in backlog)
known_ids = set(findings.threat_id)
assert all(set(row["threat_ids"]) <= known_ids for row in backlog), "backlog references a threat pytm did not raise"

out = save_json("_evidence/01_threat_backlog.json", {
    "model": tm.name,
    "findings_total": int(len(findings)),
    "llm_findings_before_controls": sorted(before),
    "llm_findings_after_controls": sorted(after),
    "backlog": backlog,
})
print("PASS: threat backlog has owners and verification methods")
print("Wrote", out.resolve())

## Review questions

- Which data flow crosses the most consequential trust boundary?
- Which control is enforced *outside* the model?
- What assumption would make this threat model false tomorrow?
- Which architecture diff should force `rag_agent_tm.py` to be reviewed?